# Batería del Campus Nord: previsión + MPC con programación lineal

Datathon UPC Sostenible 2026 · Modo Estándar y Modo Difícil. Un mismo `agent.py` detecta el modo en la primera llamada. Todo lo de este cuaderno se reproduce con los scripts de `scripts/` (ver README).

## 1. El score es lineal en la energía importada

Con la base sin batería fija, minimizar Σ w_t·red_t con **w = CO2 + λ·precio** y **λ = (0,3/0,7)·CO2_base/Coste_base** (0,2617 en Difícil, 0,2517 en Estándar) es exactamente maximizar el Score oficial. No hay pesos ajustados a mano. El rendimiento del 90 % exige una diferencia del 11,1 % entre horas; la mediana del cociente diario máx/mín de w es 2,66, así que siempre hay un ciclo diario que aprovechar. La batería no puede vender: en el LP la importación es una variable ≥ 0, y verter no premia.

## 2. Hallazgos en los datos

![El contador repite el mismo valor de 22:05 a 06:00 en 256 de 259 noches.](../reports/figures/F1_contador_congelado.png)

*El contador repite el mismo valor de 22:05 a 06:00 en 256 de 259 noches.*

![Consecuencia: de 23 h a 5 h la carga se conoce sin error; el agente lo explota (σ = 0) y gana +0,015 de forma consistente.](../reports/figures/F2_persistencia_por_hora.png)

*Consecuencia: de 23 h a 5 h la carga se conoce sin error; el agente lo explota (σ = 0) y gana +0,015 de forma consistente.*

![Solar ≈ 0,131·irradiancia (R² 0,99), con los 3 inversores recortando a 50 kW.](../reports/figures/F3_solar_vs_irradiancia.png)

*Solar ≈ 0,131·irradiancia (R² 0,99), con los 3 inversores recortando a 50 kW.*

![PVPC con escalones de la tarifa 2.0TD (P1 +0,091 €/kWh, P2 +0,023) y CO2 mínimo a mediodía: la hora más limpia coincide con el excedente solar.](../reports/figures/F4_perfiles_mercado.png)

*PVPC con escalones de la tarifa 2.0TD (P1 +0,091 €/kWh, P2 +0,023) y CO2 mínimo a mediodía: la hora más limpia coincide con el excedente solar.*

![En Estándar casi nunca hay excedente (la previsión apenas importa); en Difícil hay excedente el 9,3 % de las horas y descargar de más se pierde.](../reports/figures/F5_carga_neta.png)

*En Estándar casi nunca hay excedente (la previsión apenas importa); en Difícil hay excedente el 9,3 % de las horas y descargar de más se pierde.*

![Fines de semana, festivos y cierres (Navidad, agosto) cambian el nivel de consumo; en gris, las semanas de test (sin datos).](../reports/figures/F6_consumo_diario.png)

*Fines de semana, festivos y cierres (Navidad, agosto) cambian el nivel de consumo; en gris, las semanas de test (sin datos).*

## 3. Previsión

LightGBM directo multi-horizonte por bandas (consumo y solar h0 | h1–2 | h3–23; precio y CO2 h3–7 | h8–26), cada modelo predice la corrección sobre una base conocida. La solar se modela como índice de claridad sobre un cielo despejado empírico calculado en hora UTC. Los árboles se exportan a arrays y se evalúan con numpy (paridad exacta con LightGBM, sin pickles). Aumentos de datos para arranque en frío (noviembre no tiene sensores en train) y para el Modo Estándar sin meteo.

![Error fuera de muestra (modelos sin las semanas evaluadas) frente a referencias ingenuas: la carga neta de la hora actual baja de 118 a 59 kW; el precio a +8–23 h, de 41 a 17 ×10⁻³ €/kWh.](../reports/figures/F8_error_prevision.png)

*Error fuera de muestra (modelos sin las semanas evaluadas) frente a referencias ingenuas: la carga neta de la hora actual baja de 118 a 59 kW; el precio a +8–23 h, de 41 a 17 ×10⁻³ €/kWh.*

![Los 9 escenarios de carga neta (σ̂ condicional × cuantiles del error normalizado) están bien calibrados en las colas, que es donde se decide cuánto descargar.](../reports/figures/F9_calibracion_escenarios.png)

*Los 9 escenarios de carga neta (σ̂ condicional × cuantiles del error normalizado) están bien calibrados en las colas, que es donde se decide cuánto descargar.*

## 4. Control

Cada hora: LP de 24 h con `scipy.optimize.linprog` (HiGHS) y 9 escenarios equiprobables de carga neta con una sola acción por hora. En el óptimo, la descarga llega hasta un cuantil alto de la carga neta cuando la hora es cara y hasta uno bajo cuando es marginal. Valor terminal de la energía = cuantil 30 % de w. Al final de cada semana se planifica con la previsión del arranque de la siguiente (el SOC continúa, P2). Se ejecuta solo la primera hora y pasa por el Safety Clamp oficial.

![Una semana de abril: carga gratis con el excedente del mediodía, descarga en la punta de la tarde; el SOC del agente casi coincide con el del oráculo.](../reports/figures/F7_semana_operacion.png)

*Una semana de abril: carga gratis con el excedente del mediodía, descarga en la punta de la tarde; el SOC del agente casi coincide con el del oráculo.*

## 5. Validación

22 semanas pseudo-test (las dos últimas disponibles de cada mes) retiradas de los datos como un hueco real; dos protocolos de SOC (P1 por semana, P2 continuo como el servidor); decisiones por diferencias emparejadas por semana (> 2 SE y ≥ 60 % de semanas). Prueba de fuga automática, 0 infracciones en todos los escenarios de estrés, 27 s para 2.009 h en Difícil (límite 60 s), paquete de 1,8 MB.

In [1]:
from pathlib import Path
from IPython.display import Markdown
Markdown(Path('../reports/RESULTADOS.md').read_text(encoding='utf-8'))

# Resultados de la validación cruzada (22 semanas, modelos sin las semanas evaluadas)

P1 = cada semana por separado con SOC al 50 %; P2 = semanas seguidas con SOC continuo (protocolo del FAQ y del servidor). Score oficial agregado con base común. 0 infracciones en todo.

## T1 · Escalera de resultados

| Modo | Agente | P1 | P2 | ΔCO2 P2 | ΔCoste P2 |
| --- | --- | --- | --- | --- | --- |
| Difícil | Agente previo del equipo (README previo) | 21,73 | 19,89 | — | — |
| Difícil | **Este agente** | 21.71 | 20.34 | 19.47 % | 22.37 % |
| Difícil | Oráculo | 22.84 | 21.45 | 20.58 % | 23.49 % |
| Estándar | Agente previo (agente_anterior) | 6.33 | 5.91 | 5.55 % | 6.75 % |
| Estándar | **Este agente** | 6.33 | 5.91 | 5.59 % | 6.68 % |
| Estándar | Oráculo | 6.35 | 5.93 | 5.59 % | 6.70 % |

## T2 · Ablaciones (Difícil, diferencia emparejada frente al agente final)

| Cambio | Δ P1 | Δ P2 (± SE) | semanas ganadas P2 | Decisión |
| --- | --- | --- | --- | --- |
| Sin noche exacta (σ normal de noche) | -0.015 | -0.016 ± 0.005 | 27 % | se queda el agente final |
| σ de decisión en vez de σ por horizonte | -0.009 | -0.008 ± 0.015 | 41 % | se queda el agente final |
| Sin planificar el arranque del bloque siguiente | +0.000 | -0.383 ± 0.474 | 45 % | se queda el agente final |
| 5 escenarios en vez de 9 | -0.049 | -0.049 ± 0.016 | 23 % | se queda el agente final |
| 150 árboles por modelo en vez de 300 (2× más rápido) | -0.052 | -0.066 ± 0.071 | 32 % | se queda el agente final |

## T3 · Error de previsión fuera de muestra (Difícil, media de los pliegues A y B)

| Horizonte | Carga neta (kW) | ref. | Precio (×10⁻³) | ref. | CO2 (×10⁻³) | ref. |
| --- | --- | --- | --- | --- | --- | --- |
| h0 | 59.1 | 118.1 | — | — | — | — |
| h1-2 | 86.9 | 139.4 | — | — | — | — |
| h3 | 117.5 | 156.8 | 8.18 | 12.27 | 5.25 | 5.43 |
| h3-7 | 127.2 | 173.3 | 11.70 | 24.62 | 8.84 | 10.83 |
| h8-23 | 143.2 | 183.7 | 16.61 | 41.16 | 16.33 | 20.36 |


![Aportación de cada semana al score frente al oráculo: el hueco restante es error de previsión, concentrado en días nublados y cambios de régimen.](../reports/figures/F10_aportacion_semanal.png)

*Aportación de cada semana al score frente al oráculo: el hueco restante es error de previsión, concentrado en días nublados y cambios de régimen.*

## 6. Límites

- No se puede verificar el protocolo exacto del servidor ni el reparto del 50 % privado; el agente detecta solo si el SOC se reinicia entre semanas y si las listas traen 3 o 24 horas.
- El calendario de festivos de las semanas de test (24/09, Navidad, Semana Santa, 24/06) no se puede contrastar con datos.
- Noviembre no tiene sensores en train: esa semana arranca en frío (modelos entrenados con ese caso).
- No se usa ningún dato externo: precio, CO2 y meteo de las semanas de test solo llegan por `obs`.